In [ ]:
import pandas as pd
import numpy as np

In [13]:
year = 2024

players_reduced_df = pd.read_csv(f"procesed_data/season_player_stats_{year}_reduced/all_players.csv", encoding='utf-8')
players_historical_df = pd.read_csv(f"procesed_data/player_mean_stats_reduced.csv", encoding='utf-8')
final_dataset = pd.DataFrame()

"""
th_player1_component1_1, ..., th_player7_componentM_Z, th_bench_usg, ..., ta_player1_component1_1, ..., ta_player7_componentM_Z, ta_bench_usg, ..., th_player1_historic_component1_1, ..., 
th_player7_historic_componentM_Z, ..., ta_player1_historic_component1_1, ..., ta_player7_historic_componentM_Z, team_home_wins (1 o 0)
"""

pca_cols_current = [c for c in players_reduced_df.columns if c.startswith('PCA')]
pca_cols_historic = [c for c in players_historical_df.columns if c.startswith('PCA')]

for game_id, game_df in players_reduced_df.groupby('game_id'):
    row = {}
    row['game_id'] = game_id
    for team, team_df in game_df.groupby('team'):
        # Team home
        if team_df.iloc[0]['location'] == 'Home':
            prefix = "th"
            if team_df.iloc[0]["win"] == 1:
                team_home_wins = 1
            else:
                team_home_wins = 0
            row['team_home_wins'] = team_home_wins
        else:
            prefix = "ta"
        
        team_df = team_df.sort_values(by='mp', ascending=False)
        top5_mp = team_df.head(5)['mp'].sum()
        bench_mp = team_df['mp'].sum() - top5_mp
        row[f"{prefix}_bench_usg"] = bench_mp / top5_mp if top5_mp > 0 else 0
        # Get the 7 players with the most minutes played
        for player_idx in range(min(7, len(team_df))):  # Asegura máximo 7 jugadores
            player = team_df.iloc[player_idx]
            player_count = player_idx + 1
            for col in pca_cols_current:
                row[f"{prefix}_player{player_count}_{col}"] = player[col]
            
            # Stats históricos (con manejo de errores)
            player_name = player['player']
            hist_data = players_historical_df[players_historical_df['Player'] == player_name]
            
            for col in pca_cols_historic:
                if not hist_data.empty:
                    row[f"{prefix}_player{player_count}_historic_{col}"] = hist_data[col].values[0]
                else:
                    row[f"{prefix}_player{player_count}_historic_{col}"] = 0
    final_dataset = pd.concat([final_dataset, pd.DataFrame([row])], ignore_index=True)


# Guardar resultado
final_dataset.to_csv(f"final_data/season_players_dataset_{year}.csv", index=False, encoding='utf-8-sig')

In [ ]:
year = 2024

teams_df = pd.read_csv(f"raw_data/season_team_stats_{year}/all_teams.csv", encoding='utf-8')
teams_historical_df = pd.read_csv(f"procesed_data/all_teams_data_reduced.csv", encoding='utf-8')
final_dataset = pd.DataFrame()

"""
team_home_stat1, ..., team_home_statN, team_home_historic_component1_1, ..., team_home_historic_componentM_Z, 
team_away_stat1, ..., team_away_statN, team_away_historic_component1_1, ..., team_away_historic_componentM_Z, team_home_wins (1 o 0)
"""

cols_current = [c for c in teams_df.columns if not c.startswith('opp')]
pca_cols_historic = [c for c in players_historical_df.columns if c.startswith('PCA')] + ["Champion"]

# teams_df = teams_df.rename(columns={col: f"team_home_{col}" for col in cols_current})
# teams_df = teams_df.rename(columns={col: f"team_away_{col}" for col in cols_current})

for game_id, game_df in teams_df.groupby('game_id'):
    row = {}
    row['game_id'] = game_id
    for location, team_df in game_df.groupby('Location'):
        # Team home
        if location == 'Home':
            prefix = "team_home"
            if team_df.iloc[0]["Win"] == 1:
                team_home_wins = 1
            else:
                team_home_wins = 0
            row['team_home_wins'] = team_home_wins
        else:
            prefix = "team_away"
        
        for col in cols_current:
            row[f"{prefix}_{col}"] = team_df[col].values[0]
        
        team_historical_df = teams_historical_df[teams_historical_df['Team'] == team_df.iloc[0]['team']]
        for team_year in team_historical_df['Year'].unique():
            for col in pca_cols_historic:
                row[f"{prefix}_{str(team_year)}_historic_{col}"] = team_historical_df[col].values[0]
        
    final_dataset = pd.concat([final_dataset, pd.DataFrame([row])], ignore_index=True)


# Guardar resultado
final_dataset.to_csv(f"final_data/season_teams_dataset_{year}.csv", index=False, encoding='utf-8-sig')